RAG Pipline - Data Ingestion to Vector DB Pipeline

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

Data ingestion

In [ ]:
###Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

#Process all pdfs in the data directory
all_pdf_documents = process_all_pdfs("data")

Chunking

In [5]:
### Text splitting into chunks

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""] 
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    return split_docs

In [ ]:
chunks = split_documents(all_pdf_documents)
chunks

Embedding and vectorStoreDB

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handle embedding generation and storage using SentenceTransformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager
        
        Args: 
            model_name: HuggingFace model name for sentence embeddings
        
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded embedding model: {self.model_name}. Embedding dimension: {self.model.get_embedding_dimension()} ")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        
        Args:
            texts: List of strings to embed
        
        Returns:
            np.ndarray: Array of embeddings with shape (len(texts), embedding_dimension)
        """
        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() first.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

### Initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Vector Store

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "data/vector_store"):
        """Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "Collection of PDF document embeddings for RAG"},
                configuration={"hnsw": {"space": "cosine"}},
                )
            print(f"Initialized ChromaDB collection: {self.collection_name} at {self.persist_directory}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise e

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain document
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        "Prepare data for chromaDB"
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            #Generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{1}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)  
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(doc.metadata)

            documents_texts.append(doc.page_content)

            embeddings_list.append(emb.tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas ,
                embeddings=embeddings_list,
                documents=documents_texts
            )
            print(f"Successfully added {len(documents)} documents to the vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise e

vector_store = VectorStore()
vector_store

In [ ]:
###Convert the text to embeddings

texts = [doc.page_content for doc in chunks]

#Generate embeddings for the text chunks
embeddings = embedding_manager.generate_embeddings(texts)

###Store in the vector database
vector_store.add_documents(chunks, embeddings)

Retriever Pipeline from VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store:VectorStore, embedding_manager: EmbeddingManager):
        """ Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve top_k relevant documents for a given query
        
        Args:
            query: User query string
            top_k: Number of top documents to retrieve
            score_threshold: Minimum similarity score for retrieved documents

        Returns:
            List of dictionaries containing document metadata and content
        """

        #Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #Convert distance to similarity score (assuming distance is cosine distance)
                    similarity = 1 - (distance)
                    if similarity >= score_threshold:
                        doc_info = {
                            "similarity_score": float(similarity),
                            "metadata": metadata,
                            "content": document,
                            "id": doc_id,
                            "distance": float(distance),
                            "rank": i + 1

                        }
                        retrieved_docs.append(doc_info)
                print(f"Retrieved {len(retrieved_docs)} documents for query: '{query}'")

            else:
                print(f"No documents found for query: '{query}'")
        except Exception as e:
            print(f"Error occurred while querying the vector store: {e}")
            return []

        return retrieved_docs


rag_retriever = RAGRetriever(vector_store, embedding_manager)


In [ ]:
rag_retriever.retrieve("What is issue with current art style?")

Integration VectorDB Context pipeline with LLM Output

In [17]:
### RAG Pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

### Initialze the Groq LLM with your API key
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model="openai/gpt-oss-20b", temperature=0.1, max_tokens=500)

### Simple RAG Function to retrieve relevant documents and generate a response using the Groq LLM
def rag_query(query,retriever, llm, top_k=3):
    #retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else "No relevant documents found."
    if not context:
        
        return "No relevant documents found."

    #Generate a response using the LLM
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    response = llm.invoke(prompt.format(context=context, query=query))
    return response.content



In [ ]:
answer = rag_query("What is issue with current art style?", rag_retriever, llm, top_k=3)
print("Answer from RAG pipeline:\n", answer)

Enhanced RAG Pipeline Features

In [20]:
def rag_advanced(query, retriever, llm, top_k=3, score_threshold=0.2, return_context=False):

    """
    RAG pipeline with extra features:
    -Returns answer, sources, confidence score, and optionally full context
    """
    
    #retrieve the context
    results = retriever.retrieve(query, top_k=top_k, score_threshold=score_threshold)
    if not results:
        return {"answer": "No relevant documents found.", "sources":[], "confidence": 0.0, "context": ""}
    
    context = "\n\n".join([doc['content'] for doc in results]) if results else "No relevant documents found."
    if not context:
        return {"answer": "No relevant documents found.", "sources":[], "confidence": 0.0, "context": ""}

    sources = [
        {
            "source": doc['metadata'].get('source_file', doc['metadata'].get('source', 'Unknown')),
            "page": doc['metadata'].get('page', 'Unknown'),
            "score": doc['similarity_score'],
            "preview": doc['content'][:200] + "..."
        }
        for doc in results
    ]

    confidence = max([doc['similarity_score'] for doc in results]) if results else 0.0

    #Generate a response using the LLM
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    response = llm.invoke(prompt.format(context=context, query=query))
    output = {"answer": response.content, 
            "sources": sources,
            "confidence": confidence,
            "context": context}
    if return_context:
        output["context"] = context
    return output

In [ ]:
result = rag_advanced("What were the quantitative metrics for evaluation of computational ai artists?", rag_retriever, llm, top_k=3, score_threshold=0.2, return_context=True)
print("Answer from RAG pipeline:\n", result["answer"])
print("Sources:\n", result["sources"])
print("Context Preview:\n", result["context"][:200] + "...")
print("Confidence Score:\n", result["confidence"])